# Microsoft Foundry — agents you host, and the plumbing around them
## Content
- two hosted agents
- agent  telemetry

**Prerequisites** (details in `agent/README.md`)
1. `cd agent && azd up` — deploys `demo-hosted-agent` (Agent Framework, Responses protocol) and `demo-long-running-agent` (resilient + steerable, preview) to your existing Foundry project.
2. **Application Insights** connected to the project (already true if you ran notebook 1).
3. `.env` next to this notebook with `FOUNDRY_PROJECT_ENDPOINT`, `HOSTED_AGENT_NAME`, `LONG_RUNNING_AGENT_NAME`, `APP_INSIGHTS_RESOURCE_ID`.

## 0 · Connect
Hosted agents expose a dedicated Responses endpoint the moment they are deployed — publishing is not required — and any OpenAI-compatible SDK can call it.

Docs: learn.microsoft.com/azure/foundry/agents/concepts/hosted-agents

In [1]:
import os, time, json
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()
PROJECT_ENDPOINT  = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "https://<your-foundry-account>.services.ai.azure.com/api/projects/<your-project>")
HOSTED_AGENT      = os.getenv("HOSTED_AGENT_NAME", "demo-hosted-agent")
LONG_RUNNING      = os.getenv("LONG_RUNNING_AGENT_NAME", "demo-long-running-agent")

credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project.get_openai_client()

def agent_responses_endpoint(name):
    # documented pattern: {project_endpoint}/agents/{name}/endpoint/protocols/openai/responses
    return f"{PROJECT_ENDPOINT}/agents/{name}/endpoint/protocols/openai/responses"

def bearer():
    return credential.get_token("https://ai.azure.com/.default").token

print("Hosted agent endpoint      :", agent_responses_endpoint(HOSTED_AGENT))
print("Long-running agent endpoint:", agent_responses_endpoint(LONG_RUNNING))

Hosted agent endpoint      : https://cog-tb7tpjtuee4ji.services.ai.azure.com/api/projects/cog-tb7tpjtuee4ji-project/agents/demo-hosted-agent/endpoint/protocols/openai/responses
Long-running agent endpoint: https://cog-tb7tpjtuee4ji.services.ai.azure.com/api/projects/cog-tb7tpjtuee4ji-project/agents/demo-long-running-agent/endpoint/protocols/openai/responses


## 1 · Hosted agent — your code, Foundry's runtime

`demo-hosted-agent` is an Agent Framework agent with one function tool, running in a per-session VM-isolated sandbox with its **own Microsoft Entra agent identity**.

Hosted agents must be called through their **dedicated Responses endpoint** with a bearer token. The project's OpenAI client with an `agent_reference` is not supported for these hosted agents; that route returns HTTP 400.

The next two cells call the dedicated endpoint with different prompts. Then use the project client to look up the agent's identity and versions in the inventory.

In [2]:
import requests

resp = requests.post(
    agent_responses_endpoint(HOSTED_AGENT),
    params={"api-version": "v1"},
    headers={"Authorization": f"Bearer {bearer()}", "Content-Type": "application/json"},
    json={"input": "Which deployment type should I use for an interactive chat assistant in Switzerland?"},
    timeout=180,
 )
resp.raise_for_status()
body = resp.json()
output = body.get("output", [])
print("\n".join(
    content["text"]
    for item in output if item.get("type") == "message"
    for content in item.get("content", []) if content.get("type") == "output_text"
))
print("\nOutput items:", [item.get("type") for item in output])

The deployment hint recommends: “Provisioned (PTU) for guaranteed throughput; or GlobalStandard with Priority Processing on pay-as-you-go.”

For your interactive chat assistant, choose pay-as-you-go for variable demand or PTU for predictable, sustained traffic. If processing must stay in Switzerland, avoid Global deployments and verify that your chosen model supports a regional Standard or regional Provisioned deployment in a Swiss Azure region.

Output items: ['function_call', 'function_call_output', 'reasoning', 'message']


In [3]:
import requests

# Same agent, called on its dedicated endpoint — what an external app or another agent would do.
url = agent_responses_endpoint(HOSTED_AGENT)
resp = requests.post(url, params={"api-version": "v1"},
                     headers={"Authorization": f"Bearer {bearer()}", "Content-Type": "application/json"},
                     json={"input": "In one sentence, what are you and where are you running?"})
resp.raise_for_status()
body = resp.json()
print(body.get("status"), "—", next((o["content"][0]["text"] for o in body.get("output", []) if o.get("type") == "message"), body))

completed — I’m an AI assistant serving as a Microsoft Foundry architecture guide, accessed through an API; I don’t have visibility into the infrastructure or location hosting me.


In [4]:
# Inventory: versions, identity, protocols
versions = list(project.agents.list_versions(agent_name=HOSTED_AGENT))      # verify: list_versions signature in your SDK version
latest = versions[0]
print("Agent   :", latest.name, "version", latest.version)
print("Kind    :", getattr(latest.definition, "kind", "hosted"))
print("Identity:", getattr(latest, "agent_identity", None) or "see Control Plane → Assets → Entra ID column")

Agent   : demo-hosted-agent version 2
Kind    : hosted
Identity: see Control Plane → Assets → Entra ID column


## 2 · Procurement decision brief — background, resilient, steerable

**Business scenario:** a Swiss manufacturer must choose a sourcing strategy before a product launch in six months. The procurement manager needs a risk assessment, a mitigation plan with owners, and an executive decision brief. Each section is produced by a real model call using the preceding sections.

**Why a long-running agent?** Multi-stage analysis and review can outlast a chat request. Background execution lets the manager leave and reconnect; checkpointed sections survive a process restart; steering stops work on an obsolete decision when priorities change. A single short summary would be better served by a normal model call.

1. **Start:** prepare a preliminary battery-module sourcing brief. Create a conversation on the hosted agent's own endpoint and submit a stored background response.
2. **Steer (optional):** promptly redirect the same conversation to battery-recycling partner selection. The original turn winds down and a separate brief begins; previous-topic conclusions are not reused.
3. **Read the deliverable:** the final cell renders each section as it becomes available. Skip steering to finish the original brief. Polling can reconnect to the same response.
4. **Recover (local test):** set `SIMULATE_CRASH_AFTER_STAGE=0`, run locally, and restart after the first checkpoint. Completed sections retain their content and IDs; only uncommitted work is repeated.

**Boundaries:** this is a preliminary, model-generated brief, not verified supplier due diligence. No live supplier, price or regulatory sources are connected. Assumptions and missing evidence must be identified; a human approves decisions. A full brief normally makes three billable model calls, with additional usage possible for retries or interrupted work.

The hosted identity needs model-invocation access and `AZURE_AI_MODEL_DEPLOYMENT_NAME`. Redeploy after the refactor and start a fresh conversation. The manifest adds a **10-second demo-only delay per stage** to allow steering; set `DEMO_STAGE_DELAY_SECONDS` to `0` for normal use. Actual model latency varies.

Docs: https://learn.microsoft.com/azure/foundry/agents/how-to/deploy-resilient-agent · https://learn.microsoft.com/azure/foundry/agents/how-to/deploy-steerable-agent

In [7]:
from IPython.display import Markdown, display

LR_URL = agent_responses_endpoint(LONG_RUNNING)
API_VERSION = "2025-11-15-preview"
H = lambda: {"Authorization": f"Bearer {bearer()}", "Content-Type": "application/json"}
steer = None

def agent_request(method, url, *, required_fields=("id", "status"), **kwargs):
    response = requests.request(
        method, url, params={"api-version": API_VERSION}, headers=H(),
        timeout=180, **kwargs,
    )
    if not response.ok:
        raise requests.HTTPError(
            f"{method} {url}: HTTP {response.status_code}\n{response.text}",
            response=response,
        )
    payload = response.json()
    if not isinstance(payload, dict) or any(field not in payload for field in required_fields):
        raise ValueError(f"Unexpected agent response: {payload!r}")
    return payload

def briefing_markdown(response):
    sections = [
        content["text"]
        for item in response.get("output", []) if item.get("type") == "message"
        for content in item.get("content", []) if content.get("type") == "output_text"
    ]
    completed = sum(section.startswith(("## Risk assessment\n", "## Mitigation plan\n", "## Decision brief\n")) for section in sections)
    parts = [f"**Status: {response['status']} | Briefing sections: {completed}/3**"]
    parts.extend(sections)
    if not sections:
        parts.append("The first model-backed section is being prepared; no results are available yet.")
    if response.get("error"):
        parts.append(f"**Error:** {response['error']}")
    return Markdown("\n\n---\n\n".join(parts))

def show(rid):
    response = agent_request("GET", f"{LR_URL}/{rid}")
    display(briefing_markdown(response))
    return response

conversation = agent_request(
    "POST", f"{LR_URL.removesuffix('/responses')}/conversations",
    required_fields=("id",), json={},
)
conv_id = conversation["id"]
start = agent_request(
    "POST", LR_URL,
    json={
        "input": (
            "Prepare a preliminary sourcing decision brief for a Swiss manufacturer launching "
            "a stationary battery-storage product in six months. Compare single-source and "
            "dual-source battery-module sourcing. Priorities: continuity of supply, EU/Swiss "
            "logistics and traceability. No supplier quotes, contracts or audited data have "
            "been provided. Include risks, mitigation owners and evidence needed before approval."
        ),
        "conversation": conv_id, "store": True, "background": True,
    },
)
resp_id = start["id"]
print("Started briefing:", resp_id, "| Conversation:", conv_id)
time.sleep(3)
response = show(resp_id)

Started briefing: caresp_0d4a1469b4944bda00FRiGsv7JLBtQZSbNZvELM3fqzivvXHEn | Conversation: conv_0d4a1469b4944bda00T6V4IbPeYS78lu71C4EkxbUsW10HH58N


**Status: in_progress | Briefing sections: 0/3**

---

The first model-backed section is being prepared; no results are available yet.

In [8]:
if not isinstance(conv_id, str) or not conv_id:
    raise ValueError("Run the preceding background-response cell first to create a conversation.")

first_turn = agent_request("GET", f"{LR_URL}/{resp_id}")
if first_turn["status"] not in ("queued", "in_progress"):
    raise RuntimeError("The first turn has already stopped. Rerun the preceding cell, then steer promptly.")

steer = agent_request(
    "POST", LR_URL,
    json={
        "input": (
            "Change of priority: prepare a preliminary decision brief for the same Swiss "
            "manufacturer on selecting a battery-recycling partner, not battery-module sourcing. "
            "The launch is still in six months. Compare one regional partner versus a "
            "primary partner with a backup. Focus on traceability, continuity and evidence "
            "needed to assess EU/Swiss waste-shipment obligations. No supplier quotes, permits "
            "or audited data are available. Give mitigation owners and keep the final executive "
            "brief under 200 words. Do not assume regulatory compliance."
        ),
        "conversation": conv_id, "store": True, "background": True,
    },
)
print("Redirected briefing accepted:", steer["id"], "| Status:", steer["status"])
time.sleep(3)
print("Original sourcing brief:")
first_turn = show(resp_id)
print("Recycling-partner brief:")
response = show(steer["id"])

Redirected briefing accepted: caresp_0d4a1469b4944bda00CV7hNWQW5SbqC7m9uRyN9UYNaflsxg6t | Status: queued
Original sourcing brief:


**Status: completed | Briefing sections: 0/3**

---

Work stopped before the next section was committed. Completed sections above are preliminary, not a final decision brief. A queued steering request will run separately.

Recycling-partner brief:


**Status: in_progress | Briefing sections: 0/3**

---

The first model-backed section is being prepared; no results are available yet.

In [9]:
target_id = steer["id"] if steer else resp_id
deadline = time.monotonic() + 600
progress = display(Markdown("Waiting for briefing sections..."), display_id=True)
while True:
    response = agent_request("GET", f"{LR_URL}/{target_id}")
    progress.update(briefing_markdown(response))
    if response["status"] in ("completed", "failed", "cancelled", "incomplete"):
        break
    if time.monotonic() >= deadline:
        print("The agent is still running. Rerun this cell to reconnect to the same briefing.")
        break
    time.sleep(5)
if response["status"] in ("failed", "cancelled", "incomplete"):
    raise RuntimeError(f"Briefing {response['status']}: {response.get('error') or response.get('incomplete_details') or 'No additional details'}")

**Status: completed | Briefing sections: 3/3**

---

## Risk assessment

## Preliminary decision brief — battery-recycling partner

**Decision:** Swiss manufacturer; launch in six months. Choose one regional recycler or a primary recycler with backup. No quotes, permits or audited data are available; compliance remains unassessed. This is not verified due diligence.

**Options:** One regional partner simplifies coordination but concentrates disruption risk; proximity does not establish compliance. Primary-plus-backup offers potential resilience but requires separate qualification, shipment assessment and usable reserve capacity.

**Preliminary direction:** Pursue primary-plus-backup qualification, conditional on evidence and commercial feasibility; retain human approval.

### Five prioritized risks

| Risk | Business impact | Evidence or assumption | Validation needed / mitigation owner |
|---|---|---|---|
| 1. Shipment obligations | Blocked shipments; enforcement exposure | Waste classification, routes and authorizations unknown | **Legal/EHS:** establish chemistry, waste status, origin/destination/transit; assess applicable EU/Swiss requirements and obtain documentation |
| 2. Traceability gaps | Unsubstantiated recycling claims | No audited data | **Sustainability/Quality:** verify chain of custody, downstream facilities, mass balance and treatment records |
| 3. Service interruption | Waste accumulation; production disruption | Capacity and backup independence unverified | **Operations:** validate capacity, outage plans, switching arrangements and lawful storage limits |
| 4. Battery handling failures | Fire, injury, transport delays | Handling controls unknown | **EHS/Logistics:** assess packaging, storage, transport and emergency procedures |
| 5. Qualification delay | Six-month launch jeopardized | Quotes and permits unavailable | **Procurement:** obtain evidence, pricing and qualification milestones; escalate gaps before approval |

---

## Mitigation plan

## Preliminary decision brief — battery-recycling partner
*Not verified due diligence; human approval required.*

**Supplied facts:** Swiss manufacturer; launch in six months. No supplier quotes, permits or audited data are available. Regulatory compliance remains unassessed.

| Option | Practical trade-off |
|---|---|
| One regional partner | Simpler coordination and traceability oversight; concentrates outage risk. Proximity does not establish lawful shipment routes. |
| Primary plus backup | Potential continuity benefit; requires independently qualified routes, traceability systems and demonstrable reserve capacity. Shared downstream facilities could undermine resilience. |

**Preliminary recommendation:** Pursue primary-plus-backup qualification, conditional on evidence and commercial feasibility—not supplier approval.

**Mitigations and decision gates**
- **Month 1 — Legal/EHS:** Define chemistry, condition, waste classification and origin/destination/transit routes. Identify applicable EU/Swiss shipment obligations. **Gate:** documented route-specific assessment.
- **Months 1–3 — Procurement/Quality:** Obtain quotes, facility permissions and scope, downstream-facility identities, chain-of-custody samples, mass-balance/treatment records and audit evidence. Legal/EHS must establish required notifications, consents and shipment documentation. **Gate:** material evidence gaps resolved.
- **Months 3–4 — Operations/Logistics:** Validate capacity, backup independence, switching arrangements, handling controls, emergency plans and lawful storage limits. **Gate:** credible continuity plan.
- **Months 5–6 — Accountable executive:** Approve only after qualification and shipment prerequisites are satisfied; otherwise activate a reviewed contingency or defer affected shipments.

---

## Decision brief

## Preliminary decision brief — battery-recycling partner
*Not verified due diligence; human approval required.*

**Supplied facts:** Swiss manufacturer; launch in six months. No supplier quotes, permits or audited data are available. Regulatory compliance remains unassessed.

| Option | Trade-off |
|---|---|
| One regional partner | Simpler coordination and potentially easier traceability oversight; concentrates outage risk. Proximity does not establish compliance. |
| Primary plus backup | Potential continuity benefit, but adds qualification costs and traceability interfaces. Shared downstream facilities or unavailable reserve capacity could negate resilience. |

**Conditional recommendation:** Pursue primary-plus-backup qualification, subject to route-specific legal assessment, demonstrable independent capacity, traceability and commercial feasibility. This is not supplier approval.

**Three next actions**
1. **Legal/EHS — month 1:** Establish battery chemistry, condition, waste classification and origin/destination/transit routes; assess current EU/Swiss obligations, permit scope and validity, and required notifications, consents and shipment records.
2. **Procurement/Quality — months 1–3:** Obtain quotes, authorization evidence, downstream-facility identities, chain-of-custody samples, mass-balance/treatment records and audit evidence.
3. **Operations/Logistics — months 3–4:** Validate capacity, backup independence, switching arrangements, handling controls and lawful storage contingencies; submit readiness gaps for executive decision before launch.

**Unresolved evidence gaps:** All qualification evidence above remains outstanding; neither option’s cost, operational readiness nor regulatory acceptability is established.

## 3 · Telemetry — nothing to instrument here

Both hosted agents already emit OpenTelemetry spans: the platform injects the Application Insights connection string into the container and the protocol libraries trace by default. Your notebook code adds nothing. This cell queries Application Insights for the spans the last two steps produced, so you can show the same data that backs **Foundry portal → Traces** and **Control Plane → Assets → Traces**.

Needs `pip install azure-monitor-query` and **Log Analytics Reader** on the Application Insights resource.

Docs: learn.microsoft.com/azure/foundry/observability/how-to/trace-agent-client-side

In [ ]:
from datetime import timedelta
from azure.monitor.query import LogsQueryClient
from dotenv import dotenv_values, find_dotenv

import os
env_values = dotenv_values(find_dotenv(usecwd=True))
APP_INSIGHTS_RESOURCE_ID = (env_values.get("APP_INSIGHTS_RESOURCE_ID") or os.getenv("APP_INSIGHTS_RESOURCE_ID", "")).strip()
resource_parts = APP_INSIGHTS_RESOURCE_ID.strip("/").split("/")
if (
    len(resource_parts) != 8
    or resource_parts[0].lower() != "subscriptions"
    or resource_parts[2].lower() != "resourcegroups"
    or [part.lower() for part in resource_parts[4:7]] != ["providers", "microsoft.insights", "components"]
    or any(not part or "<" in part or ">" in part for part in resource_parts)
):
    raise ValueError(
        "APP_INSIGHTS_RESOURCE_ID must be the Application Insights component resource ID: "
        "/subscriptions/.../resourceGroups/.../providers/Microsoft.Insights/components/... "
        "Do not use the Foundry project's /connections/... ID."
    )

logs = LogsQueryClient(credential)
kql = f"""
union dependencies, requests
| where timestamp > ago(30m)
| where cloud_RoleName has "{HOSTED_AGENT}" or cloud_RoleName has "{LONG_RUNNING}" or name has "responses"
| project timestamp, cloud_RoleName, name, duration, success, operation_Id
| order by timestamp desc
| take 20
"""
result = logs.query_resource(APP_INSIGHTS_RESOURCE_ID, kql, timespan=timedelta(minutes=30))
row_count = 0
for table in result.tables:
    print(" | ".join(table.columns))
    for row in table.rows:
        print(row)
        row_count += 1
print(f"\nReturned {row_count} span(s).")
if row_count == 0:
    print("No matching spans in the last 30 minutes. Check the telemetry destination, filters and ingestion delay.")

**Optional — a local OTLP view.** Run `docker run -d -p 18888:18888 -p 4317:18889 mcr.microsoft.com/dotnet/aspire-dashboard` and start an agent locally with `azd ai agent run --no-client` and `OTEL_EXPORTER_OTLP_ENDPOINT=http://localhost:4317`; the Aspire Dashboard at http://localhost:18888 shows the same spans live. `# verify` the exact OTLP environment variable the agent-server library honours in your version.

**Next: AI gateway governance.** For API Management governance of models, tools and agents, continue with [notebook 8](foundry-demo-8-ai-gateway-governance.ipynb). This hosted-agent notebook does not require an API Management gateway.

## Close in the portal  *(slide 18)*

**Control Plane → Operate → Assets**: both hosted agents with their Entra Agent IDs, runs, token usage and estimated cost; **Traces** tab for the steering run.

### Cleanup
`cd agent && azd down` removes what the `demo` environment created (the two agents), and leaves your Foundry project and deployments untouched.